# Módulo 3 - Ejercicio 1 (Plantilla)
## Conexión a IBM Quantum, Primitives, Gestión de Cuota y Despacho Temprano a la QPU

### 1. Contexto
El ecosistema de Qiskit interactúa con hardware cuántico mediante **Qiskit Runtime Service**. La ejecución se orquesta a través de **Primitives**. En esta clase utilizaremos

- **`Sampler`**: Muestrea distribuciones probabilísticas (conteos de cadenas binarias).

### Gestión de Cuota en IBM Quantum (10 minutos mensuales de QPU):
- **Tiempo en Cola:** Es gratuito y no descuenta cuota.
- **Tiempo de ejecución en procesador:** Tiempo neto en el chip cuántico. Un circuito típico de 1024 disparos consume O(1) s de QPU.
- Por tanto, **10 minutos permiten ejecutar O(10^2) experimentos** si se aplica:

  1. Validación local previa con `StatevectorSampler` (0s de cuota).
  2. Transpilación con `optimization_level=3` para reducir profundidad y compuertas.
  3. Presupuesto razonable de disparos ($N_{\text{disparos}} = 1024$, error $< 1.5\%$).
  4. Despacho asíncrono y guardado del `job_id` sin bloquear la sesión.

**Estrategia de la sesión:**

Despachamos inmediatamente el primer trabajo del día a la QPU física de IBM al iniciar el laboratorio. Mientras avanza en la cola, completamos los ejercicios 2 y 3 en simulación local, y recuperamos los resultados de la QPU en el Ejercicio 4.

---

### 2. Referencia Rápida
| Operación | Código Qiskit |
| :--- | :--- |
| Almacenar Token | `QiskitRuntimeService.save_account(channel="ibm_cloud", token="...", overwrite=True)` |
| Instanciar Servicio | `service = QiskitRuntimeService(channel="ibm_cloud")` |
| Consultar Backend con menor cola | `backend = service.least_busy(operational=True, simulator=False)` |
| Transpilar para el Backend | `t_qc = transpile(qc, backend=backend, optimization_level=3)` |
| Despachar a la QPU | `sampler = Sampler(mode=backend); job = sampler.run([t_qc], disparos=1024)` |
| Obtener ID de seguimiento | `id_trabajo = job.job_id()` |
| Simulación local analítica | `sampler = StatevectorSampler()` |
| Medir todos los qubits | `qc.measure_all()` (crea registro clásico 'meas' automáticamente) |


In [1]:
# Importaciones necesarias
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.primitives import StatevectorSampler
print("Entorno Qiskit listo.")

Entorno Qiskit listo.


### ️ Tarea 1: Creación del Circuito Cuántico para Hardware Real
1. Crea un circuito cuántico de 3 qubits llamado `qc_ghz` con `QuantumCircuit(3, name="GHZ_3Q")`.
   * *¿Por qué no declaramos bits clásicos?* A diferencia del Módulo 2 donde definíamos `QuantumCircuit(3, 3)` y medíamos manualmente cada bit con `measure(i, i)`. Al llamar `qc.measure_all()`, Qiskit crea automáticamente un registro clásico llamado 'meas' de 3 bits y mide cada qubit en su bit correspondiente.
2. Prepara el estado entrelazado GHZ: $\frac{|000\rangle + |111\rangle}{\sqrt{2}}$ aplicando Hadamard en $q_0$ y compuertas CNOT de $0 \to 1$ y $1 \to 2$.
3. Aplica mediciones finales a todos los qubits usando `qc_ghz.measure_all()`.

In [ ]:
# TODO: Construye el circuito GHZ de 3 qubits
qc_ghz = QuantumCircuit(3, name="GHZ_3Q")

### TU CÓDIGO AQUÍ ### print(qc_ghz.draw(output='text'))

### ️ Tarea 2: Validación Local Inmediata con StatevectorSampler
Antes de enviar el circuito a la nube, validemos localmente con `StatevectorSampler` que el circuito es correcto (solo debe arrojar conteos en `'000'` y `'111'`).

In [ ]:
# TODO: Ejecuta localmente con StatevectorSampler
sampler_local = StatevectorSampler()

### TU CÓDIGO AQUÍ ### # job_local = ...
# conteos_locales = ...

# print("Conteos ideales locales:", conteos_locales)

### ️ Tarea 3:  Despacho del Primer Trabajo del Día a la QPU de IBM Quantum
Ahora, conectamos con `QiskitRuntimeService`, seleccionamos el procesador cuántico con menor cola (`least_busy`), transpilamos el circuito para ese hardware y despachamos el trabajo.

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

# Instanciamos el servicio (requiere QiskitRuntimeService.save_account previamente)
service = QiskitRuntimeService(channel="ibm_cloud")

# Seleccionamos el procesador con menor cola
backend_qpu = service.least_busy(operational=True, simulator=False, min_num_qubits=3)
print(f" Conectado a la QPU de IBM: {backend_qpu.name}")
print(f"  Número de qubits físicos: {backend_qpu.num_qubits}")

# Transpilamos el circuito
qc_transpilado = transpile(qc_ghz, backend=backend_qpu, optimization_level=3)

# Despachamos a la cola de la QPU
sampler_qpu = Sampler(mode=backend_qpu)
job_qpu = sampler_qpu.run([qc_transpilado], shots=1024)

id_trabajo = job_qpu.job_id()
print("=" * 60)
print(f" TRABAJO ENVIADO CON ÉXITO A LA COLA DE IBM QUANTUM!")
print(f"  Job ID: {id_trabajo}")
print(f"  Estado inicial: {job_qpu.status()}")
print("=" * 60)
print("Guarda este Job ID. Lo recuperaremos en el Ejercicio 4 al final de la clase.")